# Week 3: Healthcare Operations, Statistical Investigation & Insight Discovery

**AnalystLab Africa · Data Science Internship** · **Author:** Molemi Molokoane

**Dataset:** healthcare dataset pack supplied by AnalystLab Africa (encounters, procedures, patients, payers, organizations, data dictionary). The records cover one hospital, Massachusetts General Hospital, from January 2011 to February 2022. Patient names in the pack end in generated digit sequences, and the field descriptions match the documentation of [Synthea](https://github.com/synthetichealth/synthea), MITRE's open-source synthetic patient generator, so the data appears to be **synthetic**. It is still treated as if it were real patient data.

**Research question:** *How do recorded total claim costs vary across encounter classes, and is encounter duration associated with cost?*

### Workflow
1. Business understanding and success criteria
2. Data inspection
3. Data preparation
4. Exploratory data analysis
5. Hypothesis 1: claim cost differs across encounter classes
6. Hypothesis 2: longer encounters cost more within a class
7. Validation
8. Limitations
9. Findings and recommendations

## 0. Setup

Runs in Google Colab or locally. The raw data pack is **not** stored in the public GitHub repository because `patients.csv` contains names and addresses, so in a fresh Colab session the setup cell asks you to upload the six CSV files.

In [ ]:
# Core libraries
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import FuncFormatter, PercentFormatter
from scipy import stats

# Display and plotting defaults
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:,.2f}".format)
sns.set_theme(style="whitegrid")
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

RANDOM_STATE = 42  # fixed seed so the bootstrap intervals are identical on every run

# Paths: raw files in Data/, charts in Visualizations/, aggregate result tables in Outputs/
DATA_DIR, VIZ_DIR, OUT_DIR = "Data", "Visualizations", "Outputs"
for folder in (DATA_DIR, VIZ_DIR, OUT_DIR):
    os.makedirs(folder, exist_ok=True)

# Encounter classes in a fixed order (largest to smallest), used by every table and chart
CLASS_ORDER = ["ambulatory", "outpatient", "urgentcare", "emergency", "wellness", "inpatient"]

# Colours: one colour for single-series charts, a fixed categorical order for multi-series charts
BAR_COLOR = "#2a78d6"
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]
INK = "#0b0b0b"
dollars = FuncFormatter(lambda v, _: f"${v:,.0f}")

def save_fig(fig, name):
    # Save a chart to Visualizations/ and display it
    fig.savefig(os.path.join(VIZ_DIR, name), bbox_inches="tight", dpi=150)
    plt.show()

In [ ]:
# Make sure the data pack is available; in Colab, upload it if it is not
REQUIRED_FILES = ["encounters.csv", "procedures.csv", "patients.csv", "payers.csv",
                  "organizations.csv", "data_dictionary.csv"]
missing_files = [f for f in REQUIRED_FILES if not os.path.exists(os.path.join(DATA_DIR, f))]

if missing_files:
    try:
        from google.colab import files
    except ImportError:
        raise FileNotFoundError(f"Copy these files into {DATA_DIR}/ and rerun: {missing_files}")
    print("Select these files in the upload dialog:", missing_files)
    for name, content in files.upload().items():
        with open(os.path.join(DATA_DIR, name), "wb") as fh:
            fh.write(content)
    missing_files = [f for f in REQUIRED_FILES if not os.path.exists(os.path.join(DATA_DIR, f))]
    assert not missing_files, f"Still missing: {missing_files}"

print("All data files present.")

## 1. Business Understanding

**The challenge.** The hospital wants to understand where its encounter costs come from. Claim costs are recorded for every encounter, but nobody has established how much they differ between types of care (ambulatory, emergency, inpatient and so on), or whether longer encounters are reliably more expensive. Without that, budgets and cost reviews rely on averages that may be dominated by a handful of very expensive encounters.

- **Problem investigated:** how recorded total claim costs vary across encounter classes, and whether encounter duration is associated with claim cost within a class.
- **Why it matters:** encounter class is known as soon as a patient arrives, and duration is something operations teams actively manage (length of stay, clinic slot length). If cost differs strongly by class, budgets and cost reviews should be set per class. If duration tracks cost, managing time is also a lever on cost; if it does not, cost drivers lie elsewhere (procedures, medications).
- **Intended user:** hospital operations and finance managers (cost review, budgeting, capacity planning). They are not statisticians, so every result is also stated in plain language.
- **Decisions supported:** (1) whether cost targets and reviews should be set per encounter class rather than hospital-wide; (2) whether encounter duration is a useful proxy for cost when planning capacity and budgets.
- **Limitations of the evidence:**
  - One hospital only, so results cannot be compared across organisations or generalised.
  - The data appears to be synthetic, so patterns reflect the generator's rules as well as real-world behaviour.
  - `TOTAL_CLAIM_COST` is a **recorded claim amount**, not the hospital's actual cost of delivering care.
  - Medication and immunisation line items are included in the total but are not supplied as separate tables, so part of the cost cannot be explained.
  - There is no clinical severity information, so differences between classes cannot be attributed to how sick patients were.
  - Many encounters come from a small number of patients, so encounters are not independent observations.

**Variables required**

| Variable | Source | Role |
|---|---|---|
| `TOTAL_CLAIM_COST` | encounters | outcome |
| `ENCOUNTERCLASS` | encounters | grouping variable (H1) |
| `START`, `STOP` → duration in hours | encounters | explanatory variable (H2) |
| `PATIENT` | encounters | cluster identifier for repeated records |
| `BASE_ENCOUNTER_COST`, procedure `BASE_COST` | encounters, procedures | explain what makes up the total |
| `PAYER` → payer group | encounters, payers | potential confounder |
| `BIRTHDATE` → age at encounter | patients | potential confounder |

### Success criteria
The work is successful if:
1. **Independent reconciliation:** encounter counts, median claim cost per class and the number of encounters over 24 hours, recalculated with a separate script that does not use pandas, match the notebook exactly.
2. **Repeated records handled:** every hypothesis test is run at the encounter level *and* with one value per patient (or a patient-level bootstrap), and a conclusion is only stated when both agree.
3. **Effect sizes, not just p-values:** every hypothesis reports an effect size with a 95% confidence interval from a bootstrap that resamples patients.
4. **Privacy:** no names, addresses or other identifying patient fields appear in any printed output, chart or exported file (checked automatically at the end of the notebook).
5. **Reproducibility:** the notebook runs top to bottom in a fresh Colab session with identical results, and the README explains how to reproduce it.

## 2. Data Inspection

### 2.1 Load the files
`utf-8-sig` strips the byte-order mark that prefixes the first column name in every file.

In [ ]:
def read(name):
    return pd.read_csv(os.path.join(DATA_DIR, f"{name}.csv"), encoding="utf-8-sig")

encounters_raw = read("encounters")
procedures_raw = read("procedures")
patients_raw = read("patients")
payers = read("payers")
organizations = read("organizations")
dictionary = read("data_dictionary")

tables = {"encounters": encounters_raw, "procedures": procedures_raw, "patients": patients_raw,
          "payers": payers, "organizations": organizations}
shape_summary = pd.DataFrame({name: {"rows": df.shape[0], "columns": df.shape[1]}
                              for name, df in tables.items()}).T
shape_summary

### 2.2 Structure and data types
Patient rows are never displayed, only the column types.

In [ ]:
encounters_raw.head()

In [ ]:
for name, df in tables.items():
    print(f"--- {name}")
    print(df.dtypes.to_string(), "\n")

Dates (`START`, `STOP`, `BIRTHDATE`, `DEATHDATE`) are read as text and must be converted. The cost columns are already numeric. The IDs are UUID strings.

### 2.3 Missing values

In [ ]:
missing_rows = []
for name, df in tables.items():
    for col in df.columns:
        n = df[col].isna().sum()
        if n:
            missing_rows.append({"table": name, "column": col, "missing": n,
                                 "pct_missing": round(100 * n / len(df), 1)})
missing_summary = pd.DataFrame(missing_rows)
missing_summary

None of the missing values affect the research question. Each one has a structural meaning, so **nothing is imputed**:
- `REASONCODE` / `REASONDESCRIPTION` (70% of encounters, 78% of procedures) are only filled in when the encounter targeted a specific condition, according to the data dictionary.
- `DEATHDATE` is blank for patients who are alive. `SUFFIX` and `MAIDEN` only apply to some people. `MARITAL` and `ZIP` are not used.
- The `NO_INSURANCE` payer has no address because it is not an organisation.

Every variable the analysis uses (class, costs, start/stop, patient, payer, birth date) is complete.

### 2.4 Data dictionary vs actual columns
The brief warns that not every documented field is present. Field names are compared after upper-casing and replacing spaces with underscores.

In [ ]:
dictionary_fields = (dictionary.dropna(subset=["Field"])
                     .assign(field=lambda d: d["Field"].str.upper().str.replace(" ", "_")))
dict_check = []
for name, df in tables.items():
    documented = set(dictionary_fields.loc[dictionary_fields["Table"] == name, "field"])
    actual = {c.upper() for c in df.columns}
    dict_check.append({"table": name,
                       "documented but absent": sorted(documented - actual),
                       "present but undocumented": sorted(actual - documented)})
pd.DataFrame(dict_check)

`patients.csv` is missing `MIDDLE` and `FIPS County Code`. Neither is needed here. All encounter and procedure fields match the dictionary.

### 2.5 Privacy: drop identifying patient fields
`patients.csv` contains names, addresses, birthplace and coordinates. The analysis only needs the patient key, birth date (to derive age), death date and gender. Everything else is dropped now and never printed, charted or exported.

In [ ]:
PATIENT_KEEP = ["Id", "BIRTHDATE", "DEATHDATE", "GENDER"]
print("Dropped identifying columns:", [c for c in patients_raw.columns if c not in PATIENT_KEEP])
patients = patients_raw[PATIENT_KEEP].copy()
del patients_raw
tables["patients"] = patients

### 2.6 Duplicates
Exact duplicate rows and duplicate primary keys are checked first. Then two near-duplicate patterns are checked: encounters for the same patient starting at the same moment, and the same procedure code recorded twice within one encounter.

In [ ]:
duplicate_summary = pd.DataFrame({
    "exact duplicate rows": {n: df.duplicated().sum() for n, df in tables.items()},
    "duplicate Ids": {n: df["Id"].duplicated().sum() if "Id" in df else np.nan for n, df in tables.items()},
})
display(duplicate_summary)

same_start = encounters_raw.duplicated(["PATIENT", "START"], keep=False)
same_start_class = encounters_raw.duplicated(["PATIENT", "ENCOUNTERCLASS", "START"], keep=False)
same_start_full = encounters_raw.duplicated(
    ["PATIENT", "START", "STOP", "ENCOUNTERCLASS", "CODE", "TOTAL_CLAIM_COST"], keep=False)
repeated_procedures = procedures_raw.duplicated(["ENCOUNTER", "CODE", "START"], keep=False)

print(f"Encounters sharing patient + start time:              {same_start.sum():,}")
print(f"  ... and the same class:                              {same_start_class.sum():,}")
print(f"  ... and identical stop, code and claim cost:         {same_start_full.sum():,}")
print(f"Procedures repeated within an encounter (same code+start): {repeated_procedures.sum():,}")

In [ ]:
# What do the same-start encounters look like? (no patient identifiers shown)
encounters_raw[same_start].groupby(["ENCOUNTERCLASS", "DESCRIPTION"]).size().sort_values(ascending=False).head(10)

There are no exact duplicates and no repeated primary keys. Encounters that share a patient and start time all have distinct IDs. The tables above show whether they are genuinely different encounters or likely double entries; the decision is recorded in Section 3.

### 2.7 Relationships between tables
`encounters` is the central table: `PATIENT` → `patients.Id`, `PAYER` → `payers.Id`, `ORGANIZATION` → `organizations.Id`, and `procedures.ENCOUNTER` → `encounters.Id` (one encounter, zero or more procedures).

In [ ]:
key_checks = [
    ("encounters.PATIENT → patients.Id", encounters_raw["PATIENT"], patients["Id"]),
    ("encounters.PAYER → payers.Id", encounters_raw["PAYER"], payers["Id"]),
    ("encounters.ORGANIZATION → organizations.Id", encounters_raw["ORGANIZATION"], organizations["Id"]),
    ("procedures.ENCOUNTER → encounters.Id", procedures_raw["ENCOUNTER"], encounters_raw["Id"]),
    ("procedures.PATIENT → patients.Id", procedures_raw["PATIENT"], patients["Id"]),
]
pd.DataFrame([{"relationship": label, "rows": len(fk), "unmatched": (~fk.isin(pk)).sum()}
              for label, fk, pk in key_checks])

In [ ]:
print("Organisations in the data:", organizations["NAME"].tolist())
print("Encounters with at least one procedure:",
      f"{encounters_raw['Id'].isin(procedures_raw['ENCOUNTER']).sum():,} of {len(encounters_raw):,}")

All foreign keys match: there are no orphan records. Because there is **only one organisation**, hospital-level comparisons are impossible. Only about half of encounters have a procedure, so procedure cost can only partly explain claim cost.